In [6]:
# !curl -L -o yolov7.pt https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt
# !git clone https://github.com/WongKinYiu/yolov7.git
!pip install -r requirements.txt

  Using cached torch-2.10.0-cp310-cp310-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp310-cp310-win_amd64.whl.metadata (5.4 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-win_amd64.whl.metadata (2.4 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/113.7 MB 4.8 MB/s eta 0:00:24
    --------------------------------------- 1.6/113.7 MB 4.9 MB/s eta 0:00:23
    -------------

In [9]:
import sys
import os
import cv2
import torch
import numpy as np

In [10]:

yolov7_path = os.path.abspath('./yolov7')
if yolov7_path not in sys.path:
    sys.path.insert(0, yolov7_path)

from models.experimental import attempt_load
from utils.general import non_max_suppression, scale_coords
from utils.datasets import letterbox

ModuleNotFoundError: No module named 'models'

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# DEVICE = 'cpu'
WEIGHTS = 'yolov7.pt'
VIDEO_PATH = r"data\traffic.mp4"
START_TIME = 5  
END_TIME = 120
ROI_POINTS = np.array([
    [650, 717],
    [1276, 718],
    [1277, 210],
    [650, 192],
], np.int32)


print(f"Praca na urządzeniu: {DEVICE}")

In [ ]:
try:
    print("Ładowanie modelu YOLOv7...")
    ckpt = torch.load(WEIGHTS, map_location=DEVICE, weights_only=False)
    model = ckpt['ema' if ckpt.get('ema') else 'model'].float().fuse().eval()
    print("[SUCCESS] Model załadowany poprawnie!")
except Exception as e:
    print(f"[ERROR] Nie udało się załadować modelu: {e}")
    print("Upewnij się, że plik yolov7.pt znajduje się w głównym folderze.")

In [ ]:

def run_analysis():
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print(f"Błąd: Nie można otworzyć pliku wideo: {VIDEO_PATH}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(START_TIME * fps))
    end_frame = int(END_TIME * fps)

    print("Analiza w toku... Naciśnij 'q' w oknie wideo, aby przerwać.")

    while cap.isOpened():
        ret, frame0 = cap.read()
        if not ret or int(cap.get(cv2.CAP_PROP_POS_FRAMES)) > end_frame:
            break


        img = letterbox(frame0, 640, stride=32, auto=True)[0]
        img = img[:, :, ::-1].transpose(2, 0, 1)  
        img = np.ascontiguousarray(img)
        
        img_tensor = torch.from_numpy(img).to(DEVICE).float() / 255.0
        if img_tensor.ndimension() == 3:
            img_tensor = img_tensor.unsqueeze(0)

 
        with torch.no_grad():
            pred = model(img_tensor, augment=False)[0]

        pred = non_max_suppression(pred, conf_thres=0.7, iou_thres=0.45, classes=[2, 3, 5, 7])


        cv2.polylines(frame0, [ROI_POINTS], isClosed=True, color=(0, 255, 0), thickness=2)

        for i, det in enumerate(pred):
            if len(det):
                det[:, :4] = scale_coords(img_tensor.shape[2:], det[:, :4], frame0.shape).round()

                for *xyxy, conf, cls in det:
                    x1, y1, x2, y2 = map(int, xyxy)
                    cx = (x1 + x2) // 2
                    cy = y2

                    if cv2.pointPolygonTest(ROI_POINTS, (cx, cy), False) >= 0:

                        cv2.rectangle(frame0, (x1, y1), (x2, y2), (0, 0, 255), 2)
                        cv2.circle(frame0, (cx, cy), 5, (255, 0, 0), -1)

  
        cv2.imshow('YOLOv7 - Traffic Counter', frame0)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Analiza zakończona.")

if 'model' in locals():
    run_analysis()